# Phase 1 — Survival Model Analysis & Validation

## Stage III Colon Cancer Overall Survival

This notebook documents the Phase 1 survival-analysis workflow for the oncology survival simulation project.

The primary objective is to develop and validate a multivariable Cox proportional hazards model that can later be compared with a Random Survival Forest (RSF) as alternative parameterizations of baseline patient risk for the downstream microsimulation.

### Primary research question

> Does the choice of survival model — Cox proportional hazards vs. Random Survival Forest — meaningfully change simulated long-term patient outcomes?

### Phase 1 objectives

1. Describe the finalized analysis cohort and survival outcome.
2. Characterize follow-up and censoring.
3. Explore survival patterns using Kaplan-Meier methods.
4. Fit the prespecified multivariable Cox proportional hazards model.
5. Evaluate model coefficients and hazard ratios.
6. Assess the proportional hazards assumption.
7. Apply the predefined diagnostic/remedy hierarchy if necessary.
8. Validate key statistical invariants and model outputs.
9. Freeze and document the final Phase 1 model before beginning the RSF comparison.

## 1. Analysis Scope

### Baseline prognostic modeling

The Cox and RSF models are intended to represent **baseline prognostic risk** for the downstream patient-level microsimulation.

Therefore, the primary model should use information available at or near diagnosis rather than variables that occur after the patient has already entered the disease course.

This distinction is important because the purpose of the model is not to estimate the causal effect of treatment. It is to parameterize patient-level survival risk in a way that can be compared across modeling approaches.

### Primary endpoint

**Overall survival (OS)** is the primary endpoint.

- Time variable: survival time from diagnosis
- Event: death from any cause
- Censoring: alive at last known follow-up

Cause-specific death information is retained for potential sensitivity analysis and quality control but is not used as the primary event definition.

In [1]:
import numpy as np
import pandas as pd
from lifelines import CoxPHFitter
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent))

from  src.models.models import preprocess_for_modeling

In [ ]:
df = preprocess_for_modeling()

N = df["Time"].count()
mean_followup = df['Time'].mean()
median_followup = df['Time'].median()
event_counts = df['Event'].value_counts().sort_index()
censored_count = event_counts.get(0, 0)
events = event_counts.get(1, 0)

Cleaned data loaded successfully.
Cleaned data shape: (14443, 36)
Recoded 99 -> NaN in: ['Regional nodes examined (1988+)', 'Regional nodes positive (1988+)', 'RX Summ--Surg Prim Site (1998-2022)']


In [ ]:
print(f"Total number of patients: {N}")
print(f"Mean follow-up time: {mean_followup:.2f} months")
print(f"Median follow-up time: {median_followup:.2f} months")
print(f"Number of censored observations: {censored_count}")
print(f"Number of events: {events}")

Total number of patients: 14239
Mean follow-up time: 61.72 months
Median follow-up time: 53.00 months
Number of censored observations: 7832
Number of events: 6407
Event rate: 0.45


## 2. Finalized Analysis Cohort

The analysis cohort was defined before survival modeling.

### Population definition

- Primary site: colon (`C18.0–C18.9`)
- Malignant behavior
- One primary only
- Diagnosis years: 2010–2021
- Exclude autopsy/death-certificate-only reporting sources
- Stage III disease after harmonization across SEER staging eras

### Why 2010–2021?

The study period balances:

- sufficient follow-up for survival analysis,
- consistency across staging systems,
- avoidance of very recent diagnosis years with limited follow-up.

### Stage harmonization

SEER staging fields were extracted as columns rather than filtering on stage during SEER*Stat extraction.

This preserves cases with incomplete or unknown staging information so that stage harmonization and Stage III classification can occur transparently and reproducibly in Python.

### Analysis cohort

* **Cohort Size:** $14,239\text{ patients }$
* **Mean Follow-up:** $61.72\text{ months }$
* **Median Follow-up:** $53.00\text{ months}$
* **Events:** $6407$
* **Censored =** $7832$

## 3. Prespecified Cox Covariates

The primary Cox model uses the following baseline covariates:

| Covariate | Role / Representation |
|---|---|
| Age at diagnosis | Continuous |
| Sex | Binary |
| Stage III substage | Categorical; IIIA reference |
| Marital status | Categorical |
| Race | Categorical |
| Regional nodes examined | Continuous |
| Year of diagnosis | Continuous |

The covariate set was selected based on clinical relevance, population-based survival literature, and the intended role of the model as a baseline prognostic model.

The same baseline covariate set should be used when comparing the Cox model with the RSF model unless a methodological reason for deviation is explicitly documented.

### Treatment variables

Chemotherapy and surgery were extracted from SEER but are not included in the primary baseline prognostic model.

Chemotherapy is excluded because treatment receipt occurs after diagnosis and may introduce treatment-selection bias and immortal-time bias. A naive baseline coefficient for chemotherapy would therefore answer a different question and could contaminate the intended Cox-vs-RSF comparison.

A separate landmark or time-varying treatment analysis could be considered in future work, but that would constitute a different analysis.

## 4. Reference Categories

Categorical variables must have explicitly defined reference categories before interpreting Cox coefficients.

### Reference categories

| Covariate | Reference category |
|---|---|
| Sex | Female |
| Stage III substage | IIIA |
| Marital status | Single (never married) |
| Race | Black |

### Interpretation

Each categorical hazard ratio represents the estimated relative hazard compared with its defined reference category, holding the other model covariates constant.

For example:

> If IIIB has an HR of 1.40 relative to IIIA, the model estimates approximately 40% higher instantaneous hazard for IIIB compared with IIIA, conditional on the other covariates in the model.

This interpretation assumes the proportional hazards assumption is reasonably satisfied for the covariate.


## 5. Missing-Data Strategy

Missingness was evaluated at the field level because SEER special codes differ by variable.

The project does **not** use a blanket missing-value replacement across all fields.

**Method:** ran `value_counts(dropna=False)` on each of the 36 columns in the
cleaned analytic cohort (Stage III colon cancer, SEER Research Data,
2010–2023), flagged any column containing common SEER sentinel-code patterns
(`9`, `99`, `999`, `9999`, `88`, text sentinels), then verified the actual
meaning of each flagged code against the SEER Program's field documentation
(Dictionary of SEER Variables / Data Description for SEER Research and
Research Plus) rather than assuming a generic convention applies.


## Disposition Table
 
| Column | Flagged value(s) | Count | Meaning (per SEER field documentation) | Disposition |
|---|---|---|---|---|
| Survival months | `9`, `88`, `99` | 120 / 73 / 64 | Genuine duration values (9, 88, and 99 months of survival) — this field's range legitimately spans these numbers. Not a categorical sentinel field. | Kept as-is. Not missing. |
| Time (derived from Survival months) | `9`, `88`, `99` | 120 / 73 / 64 | Same as above — `Time` is a direct derivation of `Survival months`. | Kept as-is. Not missing. |
| Regional nodes examined (1988+) | `9` | 158 | Exact count: field allows `01–89` as literal node counts. `9` = 9 nodes examined. | Kept as-is. Not missing. |
| Regional nodes examined (1988+) | `88` | 2 | Exact count: `88` falls within the valid `01–89` count range (88 nodes examined) — distinct from the AJCC staging field's use of `88` as "not applicable." | Kept as-is. Not missing. |
| Regional nodes examined (1988+) | `99` | 4 | Unknown whether nodes were removed. | Recoded → NaN via `resolve_special_codes()`. |
| Regional nodes positive (1988+) | `9` | 259 | Exact count: 9 positive nodes. Same `01–89` convention as nodes examined. | Kept as-is. Not missing. |
| Regional nodes positive (1988+) | `99` | 8 | Unknown whether nodes were positive. | Recoded → NaN via `resolve_special_codes()`. |
| RX Summ--Surg Prim Site (1998-2022) | `99` | 2 | Site-specific surgery code field (codes vary by cancer site per SEER Appendix C). `99` inferred as unknown/not stated by SEER's general trailing-9s convention; not directly confirmed against the colon-specific code table for this field. | Recoded → NaN via `resolve_special_codes()`. Low impact (n=2). Flagged as an inference, not a confirmed field-specific citation — follow-up: verify against Appendix C if this field is used further. |
| Age recode with single ages and 90+ | — | 0 flagged | Field checked for missingness after `str.extract()` parsing; confirmed complete (age is a near-universally captured field in cancer registry data). | No missing values found. See `seer_field_selection_rationale.md` for the field-selection history (single-ages vs. ranged-bin recode). |
| Stage (consolidated) | `III`, `IIINOS` | 153 / 51 (204 total) | Confirmed Stage III with unspecified AJCC substage — cannot be assigned to IIIA/IIIB/IIIC. Not a missing-value sentinel in the traditional sense, but functionally unusable for substage-adjusted analysis. | Excluded from the substage-adjusted analytic cohort (n=14,239 remaining from n=14,443). See CHANGELOG 0.7.0 — this exclusion also resolved a silent reference-category contamination bug in `pd.get_dummies(..., drop_first=True)`, where `III` (alphabetically first) was being dropped as the implicit reference instead of `IIIA`. |
| Sex | — | 0 flagged | Complete; binary Female/Male values only. | No missing values found. |
| Chemotherapy recode (yes, no/unk) | — | 0 flagged as missing | Field itself combines two distinct states ("No" and "Unknown") into a single `No/Unknown` category — a SEER-defined collapsing, not something introduced by this pipeline. | Kept as SEER-defined. Documented as a modeling limitation: cannot distinguish confirmed non-receipt from unknown receipt status. See also immortal-time-bias note below. |


## 6. Survival Outcome & Censoring

### Time variable

`survival_months`

This represents the number of months from diagnosis to death or last known follow-up.

### Event indicator

For the primary overall-survival analysis:

- `event = 1` → death
- `event = 0` → censored

### Why overall survival?

Overall survival avoids ambiguity associated with cause-of-death classification and provides a common endpoint that can be modeled consistently by both Cox PH and RSF.